Source:
* https://www.datacamp.com/tutorial/knowledge-graph-rag
* https://www.npmjs.com/package/download-git-repo
* https://github.com/tomasonjo/blogs/blob/master/llm/enhancing_rag_with_graph.ipynb?ref=blog.langchain.dev

## PROCESS

1. Load files from repo
2. Create function chunking from each file
    Metadata:
        filename: ?
        line_number_mapping: ?
        function:
        function_call_stack
    Content:
3. Chunking
    TextSplitter
    https://python.langchain.com/docs/how_to/code_splitter/

STEP 2: Initialize language model
1. instantiate language model (OpenAI)
2. llm.transformer.convert_to_graph_documents

STEP 3: Store to vector database
Embeddings

STEP 4: Retrieve knowledge for RAG

Step 5: Evaluate response (langchain)

# Installation

In [1]:
! pip install --upgrade --quiet pip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 17.1 MB/s eta 0:00:00


In [3]:
%pip install --upgrade --quiet \
  GitPython \
  langchain \
  langchain-community \
  langchain-experimental \
  langchain-openai \
  langchain-text-splitters \
  langchain_experimental \
  langchain_openai \
  langchain_pinecone \
  neo4j \
  openai \
  pinecone \
  pinecone-client \
  pinecone-plugin-inference \
  pinecone-plugin-assistant \
  pygithub \
  pygments \
  pyngrok \
  python-dotenv \
  requests \
  semchunk \
  streamlit \
  tiktoken \
  tree-sitter \
  tree-sitter-language-pack \
  wikipedia \
  yfiles_jupyter_graphs

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.7/35.7 MB 44.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.0/209.0 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.6/50.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 301.7/301.7 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.6/389.6 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.8/419.8 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.8/244.8 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━

# Environment Variables

In [4]:
# This is to create .env file on Colab
# Copy and paste your existing environment variables from your .env
# below the code
%%writefile .env

Overwriting .env


# Setting Tunnels (ngrok)

In [5]:
from threading import Thread
from pyngrok import ngrok
import os
from google.colab import userdata

from dotenv import load_dotenv

load = load_dotenv()

if load:
  ngrok.set_auth_token(os.getenv('NGROK_AUTH_TOKEN'))
  print('NGROK_AUTH_TOKEN has been set')
else:
  print('NGROK_AUTH_TOKEN has not been set')

def run_streamlit():
  os.system('streamlit run /content/app.py --server.port 8501')

# Thread allows to run the programs in parallel.
def start_thread():
  thread = Thread(target=run_streamlit)
  thread.start()

def connect():
  public_url = ngrok.connect(addr='8501', proto='http', bind_tls=True)
  print('Public URL:', public_url)

def close_tunnels():
  tunnels = ngrok.get_tunnels()

  for tunnel in tunnels:
    print(f"Closing tunnel: {tunnel.public_url} -> {tunnel.config['addr']}")
    ngrok.disconnect(tunnel.public_url)

NGROK_AUTH_TOKEN has been set


# Create Directory
Manually create 'streamlit' directory and init file or run the code below to create the folder and file

In [6]:
import os

os.makedirs("app", exist_ok=True)
with open("app/__init__.py", "w") as file: pass

# Clients

In [7]:
%%writefile app/clients.py
import pinecone
from openai import OpenAI
import os
from dotenv import load_dotenv
from neo4j import GraphDatabase
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain.vectorstores import Neo4jVector
from langchain.graphs import Neo4jGraph

load = load_dotenv()

class OpenAIClient:
    def __init__(self):
        if load:
          self.client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
          print('OpenAIClient has been instantiated')
        else:
          print('Failed to instantiate OpenAIClient')

    def create_completion(self, prompt, model="gpt-3.5-turbo", max_tokens=50):
        response = self.client.Completion.create(
            engine=model,
            prompt=prompt,
            max_tokens=max_tokens,
            n=1,
            stop=None,
            temperature=0,
        )
        return response.choices[0].text.strip()


class OpenAIEmbeddingsClient:
    def __init__(self):
        if load:
            self.client = OpenAIEmbeddings(
                api_key=os.getenv("OPENAI_API_KEY"),
                model="text-embedding-3-large",
                dimensions=EMBEDDING_DIMENSIONS
            )
            print('OpenAIEmbeddingsClient has been instantiated')
        else:
            print('Failed to instantiate OpenAIEmbeddingsClient')

    def embed_documents(self, content):
        """Get embedding for the given content."""
        return self.client.embed_documents(content)



# class PineconeClient:
#     def __init__(self):
#         if load:
#             self.client = pinecone(api_key=os.getenv("PINECONE_API_KEY"))
#             print('PineconeClient has been instantiated')
#         else:
#             print('Failed to instantiate PineconeClient')

#     def create_index(self, index_name):
#         self.index_name = index_name
#         self.client.create_index(
#             name=index_name,
#             dimension=1536,
#             metric='cosine'
#         )
#     def get_index(self, index_name):
#         return self.client.get_index(index_name)

#     def delete_index(self, index_name):
#         self.client.delete_index(index_name)

#     def from_documents(self, documents, embeddings, index_name):
#         return PineconeVectorStore.from_documents(
#             documents=documents,
#             embedding=embeddings,
#             index_name=index_name
#         )

#     def retrieve(self, index_name, top_k = 2):
#       vectorstore = self.client.from_existing_index(index_name)
#       return vectorstore.as_retriever(search_kwargs={"k": top_k})


class Neo4jClient:
    def __init__(self):
        self.driver = GraphDatabase.driver(
            Config.NEO4J_URI,
            auth=(os.getenv("NEO4J_USERNAME"), os.getenv("NEO4J_PASSWORD"))
        )
        print('Neo4jClient has been instantiated')

    def run_query(self, query, parameters=None):
        parameters = parameters or {}
        with self.driver.session() as session:
            result = session.run(query, **parameters)
            return [record.data() for record in result]

    def close(self):
        self.driver.close()


Overwriting app/clients.py


# Utils

In [8]:
%%writefile app/utils.py
import os
import sys

def set_module():
    # Add PYTHONPATH to sys.path
    python_path = os.getenv("PYTHONPATH")
    # Dynamically add the parent directory of 'app' to sys.path
    project_path = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
    if project_path and project_path not in sys.path:
        sys.path.append(python_path)

Overwriting app/utils.py


# Constants

In [9]:
%%writefile app/constants.py

IGNORED_DIRS = {
    'node_modules',
    'venv',
    'env',
    'dist',
    'build',
    'vendor',
    '__pycache__',
    'pacakge.json',
    'tsconfig.json'
}

MAX_TOKENS_CHUNK_SIZE = 5000
EMBEDDING_DIMENSIONS = 1536

Overwriting app/constants.py


# Language Support

In [10]:
%%writefile app/language_support.py

class LanguageSupport:
    # Class-level dictionary to store file extensions and their languages
    lang_map = {
        ".cpp": "cpp",
        ".go": "go",
        ".java": "java",
        ".kt": "kotlin",
        ".js": "js",
        ".jsx": "js",
        ".ts": "ts",
        ".tsx": "ts",
        ".php": "php",
        ".proto": "proto",
        ".py": "python",
        ".rst": "rst",
        ".rb": "ruby",
        ".rs": "rust",
        ".scala": "scala",
        ".swift": "swift",
        ".md": "markdown",
        ".tex": "latex",
        ".html": "html",
        ".sol": "sol",
        ".cs": "csharp",
        ".cobol": "cobol",
        ".c": "c",
        ".lua": "lua",
        ".pl": "perl",
        ".hs": "haskell",
        ".ipynb": "ipynb"   # Not supported language for codeSplitter
    }

    @classmethod
    def is_supported_language(cls, extension):
        """Check if the given file extension is supported."""
        return extension in cls.lang_map

    @classmethod
    def get_language(cls, extension):
        """Get the language associated with a file extension."""
        return cls.lang_map.get(extension, "Unsupported language")

    @classmethod
    def add_language(cls, extension, language):
        """Add a new file extension and associated language."""
        cls.lang_map[extension] = language

    @classmethod
    def remove_language(cls, extension):
        """Remove a file extension from the mapping."""
        if extension in cls.lang_map:
            del cls.lang_map[extension]


Overwriting app/language_support.py


# Clone a github repo locally

In [11]:
%%writefile app/github_repo_data.py
import os
import re
import json
import requests
from git import Repo, GitCommandError
from langchain.schema import Document


class GithubRepoData:
    def __init__(self, url: str):
        """
        Initializes the GithubRepoData object with repository details.

        Args:
            url (str): The GitHub repository URL.
        """
        # Match the URL to extract the owner and repo name
        pattern = r"(?:https?://|git@)github\.com[:/](.+?)/(.+?)(?:/|\.git|$)"
        match = re.match(pattern, url)
        if not match:
            raise ValueError(f"Invalid GitHub URL: {url}")
        owner, repo = match.groups()

        # Fetch repository details from GitHub API
        api_url = f'https://api.github.com/repos/{owner}/{repo}'
        response = requests.get(api_url)
        if response.status_code == 200:
            github = response.json()
            visibility = github.get('visibility', 'unknown')

            if visibility == 'public':
                self.id = github.get('id')
                self.owner = github.get('owner').get('login')
                self.fullname = github.get('full_name')
                self.repo_name = github.get('name')
                self.main_branch = github.get('default_branch')
                self.description = github.get('description')
                self.events_url = github.get('events_url')
                self.dir_path = os.path.join(os.getcwd(), self.repo_name)
            else:
                raise ValueError(f'Repository {self.owner}/{self.repo_name} is not public.')
        elif response.status_code == 404:
            raise ValueError(f"Repository {owner}/{repo} not found.")
        elif response.status_code == 403:
            raise ValueError("GitHub API rate limit exceeded. Please try again later.")
        else:
            raise ValueError(f"Failed to fetch repository details. HTTP Status: {response.status_code}")

    def to_dict(self):
        return {
            "id": self.id,
            "owner": self.owner,
            "fullname": self.fullname,
            "repo_name": self.repo_name,
            "description": self.description,
            "events_url": self.events_url,
            "dir_path": self.dir_path,
            "url": f"https://github.com/{self.owner}/{self.repo_name}"
        }

    def exists(self) -> bool:
        """Checks if the repository is already cloned."""
        return os.path.exists(self.dir_path)

    def clone(self) -> bool:
        """Clones the repository into the specified local directory."""
        if self.exists():
            print(f"Repository {self.repo_name} already cloned at {self.dir_path}.")
            return True

        try:
            os.makedirs(self.dir_path, exist_ok=True)
            clone_url = f"https://github.com/{self.fullname}.git"
            print(f"Cloning {self.repo_name} into {self.dir_path}")
            Repo.clone_from(clone_url, self.dir_path)
            print(f"Repository {self.repo_name} cloned successfully.")
            return True
        except GitCommandError as e:
            raise ValueError(f"Failed to clone {self.repo_name}: {e}")

Overwriting app/github_repo_data.py


In [12]:
%%writefile app/local_directory_files.py
import os
from langchain.schema import Document
from app.constants import IGNORED_DIRS
from app.language_support import LanguageSupport

class LocalDirectoryFiles:
    def __init__(self, dir_path: str, repo_fullname: str, main_branch: str):
        """
        Initializes the LocalDirectoryFiles object.

        Args:
            dir_path (str): Path to the local directory.
            repo_fullname (str): Full name of the repository (e.g., owner/repo).
            main_branch (str): Main branch of the repository.
        """
        self.dir_path = dir_path
        self.repo_fullname = repo_fullname
        self.main_branch = main_branch

    def read_file(self, filepath: str) -> str:
        """Reads the contents of a file."""
        try:
            with open(filepath, "r", encoding="utf-8") as f:
                return f.read()
        except Exception as e:
            raise ValueError(f"Failed to read file {filepath}: {e}")

    def get_files(self) -> list[Document]:
        """
        Recursively retrieves all files in the directory and creates `Document` objects.

        Returns:
            list[Document]: List of Document objects.
        """
        documents = []
        for root, _, files in os.walk(self.dir_path):
            if any(ignored_dir in root for ignored_dir in IGNORED_DIRS):
                continue

            for file in files:
                file_path = os.path.join(root, file)
                extension = os.path.splitext(file_path)[1]

                if LanguageSupport.is_supported_language(extension) :
                    content = self.read_file(file_path)
                    relative_path = os.path.relpath(file_path, self.dir_path)
                    if content:
                        document = Document(
                            page_content=content,
                            metadata={
                                "filename": file,
                                "path": file_path,
                                "url": f"https://github.com/{self.repo_fullname}/blob/{self.main_branch}/{relative_path}"
                            }
                        )
                        documents.append(document)
        return documents

Overwriting app/local_directory_files.py


In [13]:
# TODO: Add persistance on already downloaded github repo
REPOS = {}
# TODO: Add persistence on already indexed github repo
REPO_LOCAL_DIRECTORIES = {}

In [14]:
REPOS

{}

In [15]:
from app.github_repo_data import GithubRepoData
from app.local_directory_files import LocalDirectoryFiles


repo1 = GithubRepoData("https://github.com/CoderAgent/SecureAgent")
repo1.clone()

if not repo1.exists:
  REPOS[repo1.fullname] = repo1

if repo1.fullname not in REPOS:
    REPOS[repo1.fullname] = repo1
    if repo1.fullname not in REPO_LOCAL_DIRECTORIES:
        REPO_LOCAL_DIRECTORIES[repo1.fullname] = LocalDirectoryFiles(repo1.dir_path, repo1.fullname, repo1.main_branch)

repo2 = GithubRepoData('https://github.com/itancio/braintumor2')
repo2.clone()

if not repo2.exists:

  REPOS[repo2.fullname] = repo2
if repo2.fullname not in REPOS:
    REPOS[repo2.fullname] = repo2
    if repo2.fullname not in REPO_LOCAL_DIRECTORIES:
        REPO_LOCAL_DIRECTORIES[repo2.fullname] = LocalDirectoryFiles(repo2.dir_path, repo2.fullname, repo2.main_branch)

localDirectoryFiles = REPO_LOCAL_DIRECTORIES['CoderAgent/SecureAgent']

documents = localDirectoryFiles.get_files()
# for doc in documents:
#     print(doc)
len(documents)


Repository SecureAgent already cloned at /content/SecureAgent.
Repository braintumor2 already cloned at /content/braintumor2.


14

# Parser/ Chunker

In [16]:
%%writefile app/chunker.py
from langchain_experimental.text_splitter import SemanticChunker

from langchain_text_splitters import (
    Language,
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter
)
import json
from langchain.schema import Document
from app.clients import OpenAIEmbeddingsClient
from app.language_support import LanguageSupport


class BaseChunkingStrategy:
    """Base class with shared chunk size, overlap, and a default splitter."""
    chunk_size = 1024
    chunk_overlap = 200
    name = "BaseChunkingStrategy"

    @classmethod
    def splitter(cls):
        """Lazily initialize and return the default splitter."""
        return CharacterTextSplitter(
            chunk_size=cls.chunk_size,
            chunk_overlap=cls.chunk_overlap,
            add_start_index=True
        )

    @classmethod
    def create_documents(cls, doc):
        content = doc.page_content
        metadata = doc.metadata
        """Split text using the default splitter."""
        return cls.splitter().create_documents(
            [content],
            [metadata]
        )


class SemanticChunkingStrategy(BaseChunkingStrategy):
    """Semantic chunking strategy with a shared splitter."""
    def __init__(self):
        self.name = "SemanticChunkingStrategy"
        self.splitter = SemanticChunker(
            OpenAIEmbeddingsClient(),
            breakpoint_threshold_type="percentile",
            add_start_index=True
        )

    def create_documents(self, doc):
        content = doc.page_content
        metadata = doc.metadata
        try:
            return self.splitter.create_documents(
                [content],
                [metadata]
            )
        except Exception:
            # Fallback to default splitter
            return BaseChunkingStrategy.create_documents(doc)


class CodeChunkingStrategy(BaseChunkingStrategy):
    """Code chunking strategy with instance-specific language."""
    def __init__(self, language=None):
        self.name = "CodeChunkingStrategy"
        self.language = language
        if language:
            self.splitter = RecursiveCharacterTextSplitter.from_language(
                language=self.language,
                chunk_size=BaseChunkingStrategy.chunk_size,
                chunk_overlap=BaseChunkingStrategy.chunk_overlap,
                add_start_index=True
            )
        else:
            self.splitter = BaseChunkingStrategy.splitter()

    def create_documents(self, doc):
        content = doc.page_content
        metadata = doc.metadata
        try:
            return self.splitter.create_documents(
                [content],
                [metadata]
            )
        except Exception:
            # Fallback to default splitter
            return BaseChunkingStrategy.create_documents(doc)


class IpynbChunkingStrategy(BaseChunkingStrategy):
    """Chunking strategy for Python Notebook."""

    def __init__(self):
        self.name = "IpynbChunkingStrategy"
        self.splitter = CodeChunkingStrategy(language="python")

    def preprocess(self, doc):
        metadata = doc.metadata
        content = doc.page_content
        json_data = json.loads(content)
        cells = json_data.get('cells', [])

        # Separate code and markdown cells
        code_cells = ['\n'.join(cell['source']) for cell in cells if cell['cell_type'] == 'code']
        markdown_cells = ['\n'.join(cell['source']) for cell in cells if cell['cell_type'] == 'markdown']

        return {
            "code": Document(page_content='\n'.join(code_cells), metadata=metadata),
            "markdown": Document(page_content='\n'.join(markdown_cells), metadata=metadata)
        }

    def create_documents(self, doc):
        processed_docs = self.preprocess(doc)
        code_doc = processed_docs['code']
        markdown_doc = processed_docs['markdown']

        try:
            # Split code using CodeChunkingStrategy
            code_chunks = self.splitter.create_documents(code_doc)
            # Use BaseChunkingStrategy for markdown
            markdown_chunks = CodeChunkingStrategy(language="markdown").create_documents(markdown_doc)
            return code_chunks + markdown_chunks
        except Exception:
            # Fallback to default splitter
            return BaseChunkingStrategy.create_documents(doc)


class ChunkingManager:
    """Manager to select appropriate chunking strategy."""
    def __init__(self):
        self.code_splitter = lambda lang: CodeChunkingStrategy(lang)
        self.ipynb_splitter = IpynbChunkingStrategy()
        self.semantic_splitter = SemanticChunkingStrategy()

    def get_splitter(self, language):
        """Return the appropriate splitter based on file type."""
        if language == "ipynb":
            splitter = self.ipynb_splitter
            print(f"Splitter adopted: {splitter.name}")
            return splitter
        elif language:
            splitter = self.code_splitter(language)
            print(f"Splitter adopted: {splitter.name} for language {language}")
            return splitter
        else:
            splitter = self.semantic_splitter
            print(f"Splitter adopted: {splitter.name} (default for unsupported file type)")
            return splitter

    def create_documents(self, doc):
        metadata = doc.metadata
        filename = metadata.get('filename')
        extension = '.' + filename.split('.')[-1]
        if LanguageSupport.is_supported_language(extension):
            lang = LanguageSupport.get_language(extension)
        else:
            lang = None
        print('language: ', lang)
        """Create documents using the appropriate splitter."""
        splitter = self.get_splitter(lang)
        return splitter.create_documents(doc)

Overwriting app/chunker.py


In [17]:
from app.chunker import ChunkingManager
chunker = ChunkingManager()
chunks = chunker.create_documents(documents[1])
chunks[0]
documents[1]

NameError: name 'EMBEDDING_DIMENSIONS' is not defined

# Embedding


In [24]:
from app.clients import OpenAIEmbeddingsClient

model = OpenAIEmbeddingsClient()
print(dir(model))

embeddings = []
chunks = []

for document in documents:
    chunked_docs = chunker.create_documents(document)
    chunks.extend(chunked_docs)
    for chunk in chunked_docs:
        content = chunk.page_content
        embedded_content = model.get_embeddings(content)
        embeddings.append(embedded_content)

OpenAIEmbeddingsClient has been instantiated
['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', 'client', 'get_embeddings']
language:  markdown
Splitter adopted: CodeChunkingStrategy for language markdown
language:  ts
Splitter adopted: CodeChunkingStrategy for language ts
language:  ts
Splitter adopted: CodeChunkingStrategy for language ts
language:  ts
Splitter adopted: CodeChunkingStrategy for language ts
language:  ts
Splitter adopted: CodeChunkingStrategy for language ts
language:  ts
Splitter adopted: CodeChunkingStrategy for language ts
language:  ts
Splitter adopted: CodeChunkingStrategy for language ts
language:  ts
Splitter adopted: CodeChunkingStrategy for language ts
language:  ts
Spli

In [ ]:
# Temporarily write embeddings and chunks in a file

with open('embeddings.txt', 'w') as f:
    f.write(f'{embeddings}')
with open('chunks.txt', 'w') as f:
    f.write(f'{chunks}')

# Temporarily read embeddings and chunks in a file
with open('embeddings.txt', 'r') as f:
    embeddings_f = f.read().strip()
with open('chunks.txt', 'r') as f:
    chunks_f = f.read().strip()

In [ ]:
print('embeddings: ',len(embeddings), len(embeddings[0]))
print('chunks: ',len(chunks), len(chunks[0]))

print('embeddings: ',len(embeddings_f), len(embeddings_f[0]))
print('chunks: ',len(chunks_f), len(chunks_f[0]))

# VectorStore

In [35]:
!pip install --upgrade

In [1]:
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone
from langchain_openai.embeddings import OpenAIEmbeddings
import os
from dotenv import load_dotenv
import time

load = load_dotenv()

class PineconeClient:
    def __init__(self,
                 index_name='codebase-rag',
                 namespace="https://github.com/CoderAgent/SecureAgent",
                 embedding=OpenAIEmbeddings()):
        if load:
            self.client = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
            self.index_name = index_name
            self.namespace = namespace
            self.embedding = embedding
            self.vectorstore = None

            existing_indexes = [index_info["name"] for index_info in self.client.list_indexes()]
            if self.index_name not in existing_indexes:
                self.client.create_index(
                    name=self.index_name,
                    dimension=EMBEDDING_DIMENSIONS,
                    metric="cosine"
                )
                while not self.client.describe_index(self.index_name).status["ready"]:
                    time.sleep(1)

            self.index = self.client.Index(self.index_name)
            self.vectorstore = PineconeVectorStore(
                index_name=self.index_name,
                embedding=self.embedding,
                namespace=self.namespace
            )
            print('PineconeClient has been instantiated')
        else:
            print('Failed to instantiate PineconeClient')

    def from_documents(self, documents):
        """
        Initializes a new vector store from a list of documents.
        Clears and replaces any existing content.
        """
        self.vectorstore = PineconeVectorStore.from_documents(
            documents=documents,
            embedding=self.embedding,
            index_name=self.index_name,
            namespace=self.namespace
        )
        print('Vector store initialized with new documents.')

    def add_documents(self, documents):
        """
        Adds new documents or vectors to an already initialized vector store.
        Assumes that the vector store is already set up and populated.
        """
        if not self.vectorstore:
            print("Vector store does not exist. Initializing a new one with provided documents.")
            self.from_documents(documents)
        else:
            self.vectorstore.add_documents(documents)
            print('Documents added to the existing vector store.')

    def retrieve(self, top_k=2):
        """
        Retrieves top_k documents from the vector store.
        """
        if self.vectorstore:
            retriever = self.vectorstore.as_retriever(search_kwargs={"k": top_k})
            return retriever
        else:
            print("Vector store is not initialized.")
            return None

    def delete_index(self):
        """
        Deletes the index associated with this client.
        """
        self.client.delete_index(self.index_name)
        print(f'Index {self.index_name} deleted.')


In [2]:
pc = PineconeClient()
pc.add_documents([documents[0]])


PineconeClient has been instantiated


NameError: name 'documents' is not defined

# GRAPH RAG

In [ ]:
%pip install neo4j yfiles_jupyter_graphs

In [ ]:
from langchain_experimental.graph_transformers import LLMGraphTransformer
from neo4j import GraphDatabase
from yfiles_jupyter_graphs import GraphWidget
from langchain_community.vectorstores.neo4j_vector import remove_lucene_chars
from langchain_community.graphs import Neo4jGraph
from langchain_openai import ChatOpenAI

graph = Neo4jGraph(
    url=neo4j_uri,
    username=neo4j_username,
    password=neo4j_password)
llm = ChatOpenAI(
    temperature=0,
    model_name="gpt-3.5-turbo-0125") # gpt-4-0125-preview occasionally has issues
llm_transformer = LLMGraphTransformer(llm=llm)

graph_documents = llm_transformer.convert_to_graph_documents(chunks)
graph.add_graph_documents(
    graph_documents,
    baseEntityLabel=True,
    include_source=True
)

In [ ]:
# directly show the graph resulting from the given Cypher query
default_cypher = "MATCH (s)-[r:!MENTIONS]->(t) RETURN s,r,t LIMIT 50"

def showGraph(cypher: str = default_cypher):
    # create a neo4j session to run queries
    driver = GraphDatabase.driver(
        uri = neo4j_uri,
        auth = (neo4j_username,
                neo4j_password))
    session = driver.session()
    widget = GraphWidget(graph = session.run(cypher).graph())
    widget.node_label_mapping = 'id'
    #display(widget)
    return widget

showGraph()

# Storing in the the vectorstore

# Retrieval and reranking using nvidia

In [ ]:
from langchain_community.vectorstores import Neo4jVector

# Unstructured data retriever
vector_index = Neo4jVector.from_existing_graph(
    OpenAIEmbeddings(),
    url=neo4j_uri,
    username=neo4j_username,
    password=neo4j_password,
    search_type="hybrid",
    node_label="Document",
    text_node_properties=["text"],
    embedding_node_property="embedding"
)


In [ ]:
# Retriever
from langchain_core.pydantic_v1 import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from typing import Tuple, List, Optional

graph.query(
    "CREATE FULLTEXT INDEX entity IF NOT EXISTS FOR (e:__Entity__) ON EACH [e.id]")

# Extract entities from text
class Entities(BaseModel):
    """Identifying information about entities."""

    names: List[str] = Field(
        ...,
        description="All the person, organization, or business entities that "
        "appear in the text",
    )

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are extracting organization and person entities from the text.",
        ),
        (
            "human",
            "Use the given format to extract information from the following "
            "input: {question}",
        ),
    ]
)

entity_chain = prompt | llm.with_structured_output(Entities)

In [ ]:
# Test
entity_chain.invoke({"question": "What methods do you have"}).names

In [ ]:
def generate_full_text_query(input: str) -> str:
    """
    Generate a full-text search query for a given input string.

    This function constructs a query string suitable for a full-text search.
    It processes the input string by splitting it into words and appending a
    similarity threshold (~2 changed characters) to each word, then combines
    them using the AND operator. Useful for mapping entities from user questions
    to database values, and allows for some misspelings.
    """
    full_text_query = ""
    words = [el for el in remove_lucene_chars(input).split() if el]
    for word in words[:-1]:
        full_text_query += f" {word}~2 AND"
    full_text_query += f" {words[-1]}~2"
    return full_text_query.strip()

# Fulltext index query
def structured_retriever(question: str) -> str:
    """
    Collects the neighborhood of entities mentioned
    in the question
    """
    result = ""
    entities = entity_chain.invoke({"question": question})
    for entity in entities.names:
        response = graph.query(
            """CALL db.index.fulltext.queryNodes('entity', $query, {limit:2})
            YIELD node,score
            CALL {
              WITH node
              MATCH (node)-[r:!MENTIONS]->(neighbor)
              RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output
              UNION ALL
              WITH node
              MATCH (node)<-[r:!MENTIONS]-(neighbor)
              RETURN neighbor.id + ' - ' + type(r) + ' -> ' +  node.id AS output
            }
            RETURN output LIMIT 50
            """,
            {"query": generate_full_text_query(entity)},
        )
        result += "\n".join([el['output'] for el in response])
    return result

In [ ]:
print(structured_retriever("what is Reviewchanges "))

In [ ]:
# Final retriever
def retriever(question: str):
    print(f"Search query: {question}")
    structured_data = structured_retriever(question)
    unstructured_data = [el.page_content for el in vector_index.similarity_search(question)]
    final_data = f"""Structured data:
{structured_data}
Unstructured data:
{"#Document ". join(unstructured_data)}
    """
    return final_data

# Defining the RAG chain

In [ ]:
from langchain_core.prompts.prompt import PromptTemplate
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.runnables import (
    RunnableBranch,
    RunnableLambda,
    RunnableParallel,
    RunnablePassthrough,
)
from langchain_core.output_parsers import StrOutputParser

# Condense a chat history and follow-up question into a standalone question
_template = """Given the following conversation and a follow up question, rephrase the follow up question to be a standalone question,
in its original language.
Chat History:
{chat_history}
Follow Up Input: {question}
Standalone question:"""  # noqa: E501
CONDENSE_QUESTION_PROMPT = PromptTemplate.from_template(_template)

def _format_chat_history(chat_history: List[Tuple[str, str]]) -> List:
    buffer = []
    for human, ai in chat_history:
        buffer.append(HumanMessage(content=human))
        buffer.append(AIMessage(content=ai))
    return buffer

_search_query = RunnableBranch(
    # If input includes chat_history, we condense it with the follow-up question
    (
        RunnableLambda(lambda x: bool(x.get("chat_history"))).with_config(
            run_name="HasChatHistoryCheck"
        ),  # Condense follow-up question and chat into a standalone_question
        RunnablePassthrough.assign(
            chat_history=lambda x: _format_chat_history(x["chat_history"])
        )
        | CONDENSE_QUESTION_PROMPT
        | ChatOpenAI(temperature=0)
        | StrOutputParser(),
    ),
    # Else, we have no chat history, so just pass through the question
    RunnableLambda(lambda x : x["question"]),
)

In [ ]:
template = """Answer the question based only on the following context:
{context}

Question: {question}
Use natural language and be concise.
Answer:"""
prompt = ChatPromptTemplate.from_template(template)

chain = (
    RunnableParallel(
        {
            "context": _search_query | retriever,
            "question": RunnablePassthrough(),
        }
    )
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
chain.invoke({"question": "Show the reviewChanges"})

In [ ]:
chain.invoke({"Show the reviewChanges code"})

# Streamlit Chat
**SOURCE**:
* [Build llm chat app](https://blog.streamlit.io/langchain-tutorial-1-build-an-llm-powered-app-in-18-lines-of-code/)

In [ ]:
import streamlit as st
from app.clients import OpenAI

st.title('Codebase Interaction')

client = OpenAI()

# Main Streamlit App

* add a required file `__init__.py` on the project directory
* In the `.env`, add `PYTHONPATH = app`. This will allow the pythonpath to point to the parent directory of the streamlit `app` folder.
* Load the `.env` file in python
```
pip install python-dotenv
```
In the load_configuration, add this code:
```
# Add PYTHONPATH to sys.path
python_path = os.getenv("PYTHONPATH")
if python_path and python_path not in sys.path:
    sys.path.append(python_path)
```

Now you can import modules from the app package
```from streamlit.constants import IGNORED_DIRS```

# Run Streamlit Dev

In [59]:
run_streamlit()
start_thread()
connect()

Public URL: NgrokTunnel: "https://47b0-34-91-225-162.ngrok-free.app" -> "http://localhost:8501"


In [61]:
close_tunnels()


Closing tunnel: https://47b0-34-91-225-162.ngrok-free.app -> http://localhost:8501
Closing tunnel: https://34c8-34-91-225-162.ngrok-free.app -> http://localhost:8501
